Task: Sample a subset of headlines, clean them using your preprocessing engine, and use `gensim` to train an Unsupervised Latent Dirichlet Allocation (LDA) model.
- Output: Print out the top dominant keywords for your discovered topics and explain if a human linguist can easily infer a clear thematic title from the mathematical clusters.

Use your custom `maha.txt` and `2e.txt` as three unique documents.

In [1]:
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
import re

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [3]:

files = ["2e.txt", "maha.txt"]

texts = []
for file in files :
  with open(file, 'r') as f:
    text = f.read()
    texts.append(text)



In [4]:
corpus = []

for text in texts:
  clean = re.sub(r'[^\w\s]', '', text)

  clean = re.sub(r'\d+', '', clean)

  tokens = word_tokenize(clean)

  tokens = [token.lower() for token in tokens]
  print(f"Before removing stop words : {len(tokens)}")

  sw = set(stopwords.words('english'))
  custom_noise = {'first', 'second', 'onethird', 'constant', 'efforts', 'picture', 'dominated', 'tends', 'months'}
  sw.update(custom_noise)


  tagged_tokens = nltk.pos_tag(tokens)

  cleaned_words = []
  for word, tag in tagged_tokens:

        # Filter out numbers, punctuation, and stop words
      if word.isalpha() and word not in sw:
          # HEAVY FILTER: Keep ONLY Nouns (NN, NNS, NNP) and Adjectives (JJ, JJR, JJS)
          # This completely destroys supporting verbs (MD, VB, VBD, VBG, VBN, VBP, VBZ)
          if tag.startswith('NN') or tag.startswith('JJ'):
              #cleaned_words = [].append(word) or word # Safe append trick
              cleaned_words.append(word)

    # Remove duplicates from the append logic loop
  cleaned_words = [w for w in cleaned_words if w]


  tokens = [token for token in tokens if token not in sw ]
  print(f"After removing stop words : {len(cleaned_words)}")

  # chunking
  chunk_size = 100
  token_limit = 500

  for i in range(int(token_limit/chunk_size)):
    corpus.append(cleaned_words[i*chunk_size:(i+1)*chunk_size])

Before removing stop words : 1649
After removing stop words : 678
Before removing stop words : 12205
After removing stop words : 6046


### LDA

In [5]:
import random

class LDA:

    def __init__(self , no_topics:int , corpus):

        self.word_topic_table = None
        self.doc_topic_table = None

        self.no_topics = no_topics
        self.topic_list = [ i for i in range(no_topics)]

        self.no_docs = len(corpus)

        self.vocab = list(set([word for doc in corpus for word in doc ]))

        self.probability_table = { word : [0]*no_topics for word in self.vocab}

        self.corpus = corpus

        self.assignment = [ [ 0 for word in doc] for doc in corpus]

        self.probability_table = { word : [0]*self.no_topics for word in self.vocab}

        self.show_table = None


    def doc_topic(self):
        table = {f"doc{i}" : [0]*self.no_topics for i in range(self.no_docs)}
        return table

    def word_topic(self):
        table = {f"{word}" : [0]*self.no_topics for word in self.vocab}
        return table

    def initialize(self):

        self.word_topic_table = self.word_topic()
        self.doc_topic_table = self.doc_topic()

        for i,doc in enumerate(self.corpus):
            for j ,word in enumerate(doc) :
                self.assignment[i][j] = random.choice(self.topic_list)

    def fill_tables(self):
        for i , doc in enumerate(self.corpus):
            for j ,word in enumerate(doc) :
                topic = self.assignment[i][j]
                self.word_topic_table[word][topic] += 1
                self.doc_topic_table[f"doc{i}"][topic] += 1


    def normalize(self , prob : list):
        total = sum(prob)
        prob_list = [ i/total for i in prob]
        return prob_list

    def sample(self , choice:list , weight:list):
        weight = self.normalize(weight)
        choice = random.choices(choice , weights = weight)
        return int(choice[0]) , weight


    def iteration(self , alpha = 0.1 , beta=0.01):

        for i , doc in enumerate(self.corpus):
            for j , word in enumerate(doc):

                current_topic = self.assignment[i][j]

                # Decreament step
                self.word_topic_table[word][current_topic] -= 1

                self.doc_topic_table[f"doc{i}"][current_topic]-= 1

                # temporary list to store probability of word getting reassigned a topic
                probability_list = []
                # Re-assigning word to topic
                for topic in range(self.no_topics):

                    # Values
                    # count of word 'w' in topic 'k'
                    n_wk = self.word_topic_table[word][topic]

                    #total number of words in topic k
                    n_k = sum([self.doc_topic_table[document][topic] for document in self.doc_topic_table])

                    # count of topic k in document d
                    n_kd = self.doc_topic_table[f"doc{i}"][topic]

                    # total words in document d
                    n_d = len(doc)

                    # total number of words in corpus
                    v = len(self.vocab)

                    # formula
                    probability = (n_wk + beta)/(n_k + v * beta) * (n_kd + alpha)/(n_d + self.no_topics *alpha)
                    probability_list.append(probability)

                # Normalization and sampling
                sampled_topic , prob = self.sample(self.topic_list , weight=probability_list)

                # adding
                self.word_topic_table[word][sampled_topic] += 1
                self.doc_topic_table[f"doc{i}"][sampled_topic] += 1

                # re-assigning
                self.assignment[i][j] = sampled_topic

                # fill the probability table
                self.probability_table[word] = prob


    def model_topics(self , iterations = 1500):

        self.initialize()
        self.fill_tables()
        for _ in range(iterations):
            self.iteration()

        self.create_prob_table()
        print("Completed Topic Modeling!!! \nYou can see by using 'show_topics()' function.")

    def create_prob_table(self):
        new_table = {f"topic{i}" : [0]*len(self.vocab) for i in range(self.no_topics)}

        for i , word in enumerate(self.probability_table):
            for j , prob in enumerate(self.probability_table[word]):
                    new_table[f"topic{j}"][i] = f"{prob:.5f}*{word}"

        self.show_table = new_table

    def show_topics(self,top_ten=True):

        for i,topic in enumerate(self.show_table):
            print(f"Topic {i+1}\n : ")
            print(sorted(self.show_table[topic][:10],reverse=True)) if (top_ten == True) else print(sorted(self.show_table[topic],reverse=True))
            print("\n")
            print("------------+------------")

    def help(self):
        print("This is help function created to guide you to use LDA class\n")
        print("\n")
        print('Functions : \n')
        text = """
        lda.model_topic() --> takes one parameter epoches , works even left blank.
        lda.show_topic() ---> Shows probability of a word with respect to topic .

        How to use :


        no_topics = 2 ( or any number you want )
        #tokenized data
        corpus = [
                ["apple", "banana", "apple", "fruit", "fruit", "banana"],
                ["dog", "cat", "dog", "animal", "pet", "cat"],
                ["banana", "fruit", "apple", "orange", "fruit", "banana"]
                ]

        lda = LDA(no_topics=no_topics , corpus=corpus)
        lda.model_topics()
        lda.show_topics()
        """
        print(text)



### Topic Modeling on Corpus

In [6]:
no_topics = 2

In [7]:
lda = LDA(no_topics=no_topics , corpus=corpus)

In [8]:
lda.model_topics()

Completed Topic Modeling!!! 
You can see by using 'show_topics()' function.


In [9]:
lda.show_topics()

Topic 1
 : 
['0.99999*class', '0.99892*educational', '0.98822*inconsistent', '0.97761*harder', '0.97752*terminology', '0.96697*easy', '0.91533*role', '0.01044*maurya', '0.00094*daman', '0.00016*india']


------------+------------
Topic 2
 : 
['0.99984*india', '0.99906*daman', '0.98956*maurya', '0.08467*role', '0.03303*easy', '0.02248*terminology', '0.02239*harder', '0.01178*inconsistent', '0.00108*educational', '0.00001*class']


------------+------------
